# Run DeepMD inference against a reference trajectory

This notebook evaluates one labelled ASE trajectory against one or more frozen DeepMD models.
Each model is written to its own output subdirectory so several trainings can be compared directly.

## Outputs
- `<output>/<model_label>/energies_forces.txt`: per-frame DFT vs DeepMD energies
- `<output>/<model_label>/energies_forces_atoms.txt`: per-atom DFT vs DeepMD force components


In [1]:
from pathlib import Path

from ase.io import read
from deepmd.infer import DeepPot
from tqdm import tqdm

try:
    from ase.calculators.deepmd import DeepMDCalculator as DP
except Exception:
    from deepmd.calculator import DP


## Configuration


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "model_analysis" else Path.cwd().resolve()

MODEL_DIR = PROJECT_ROOT / "model_analysis" / "model"
MODEL_PATTERN = "**/*.pb"
INPUT_DIR = PROJECT_ROOT / "model_analysis" / "input"
TRAJECTORY_PATTERN = "*.traj"
TRAJECTORY_INDEX = 0
OUTPUT_ROOT = PROJECT_ROOT / "model_analysis" / "outputs" / "inference"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [3]:
model_files = sorted(MODEL_DIR.glob(MODEL_PATTERN))
if not model_files:
    raise FileNotFoundError(f"No model file matched '{MODEL_PATTERN}' in {MODEL_DIR}")

trajectory_files = sorted(INPUT_DIR.glob(TRAJECTORY_PATTERN))
if not trajectory_files:
    raise FileNotFoundError(f"No trajectory file matched '{TRAJECTORY_PATTERN}' in {INPUT_DIR}")
if TRAJECTORY_INDEX >= len(trajectory_files):
    raise IndexError(f"TRAJECTORY_INDEX={TRAJECTORY_INDEX} but only {len(trajectory_files)} file(s) were found.")

def model_label(path: Path) -> str:
    return "__".join(path.relative_to(MODEL_DIR).parts).replace(" ", "_")

print("Matched model files:")
for idx, path in enumerate(model_files):
    print(f"  [{idx}] {path.relative_to(MODEL_DIR)} -> {model_label(path)}")

trajectory_file = trajectory_files[TRAJECTORY_INDEX]
print(f"Selected trajectory: {trajectory_file.name}")
frames = read(str(trajectory_file), index=":")
print(f"Loaded {len(frames)} frames")


Matched model files:
  [0] graph.001.pb -> graph.001.pb
  [1] graph.002.pb -> graph.002.pb
  [2] graph.003.pb -> graph.003.pb
  [3] graph.004.pb -> graph.004.pb
Selected trajectory: TEST_selected.traj
Loaded 2000 frames


## Evaluate all models


In [4]:
reports = []

for model_file in model_files:
    label = model_label(model_file)
    output_dir = OUTPUT_ROOT / label
    output_dir.mkdir(parents=True, exist_ok=True)

    model = DeepPot(str(model_file))
    calculator = DP(model=str(model_file), type_map=model.get_type_map())

    frame_output = output_dir / "energies_forces.txt"
    atom_output = output_dir / "energies_forces_atoms.txt"

    with frame_output.open("w", encoding="utf-8") as fh_energy, atom_output.open("w", encoding="utf-8") as fh_atom:
        fh_energy.write("# frame energy_DFT_eV energy_DP_eV\n")
        fh_atom.write("# frame atom element fx_DFT fy_DFT fz_DFT fx_DP fy_DP fz_DP energy_DFT energy_DP\n")

        for frame_index, atoms in enumerate(tqdm(frames, total=len(frames), desc=f"Evaluating {label}")):
            atoms_ref = atoms.copy()
            atoms_ref.calc = atoms.calc
            atoms_dp = atoms.copy()
            atoms_dp.calc = calculator

            energy_ref = atoms_ref.get_potential_energy()
            forces_ref = atoms_ref.get_forces()
            energy_dp = atoms_dp.get_potential_energy()
            forces_dp = atoms_dp.get_forces()

            fh_energy.write(f"{frame_index:6d} {energy_ref: .10f} {energy_dp: .10f}\n")
            for atom_index, symbol in enumerate(atoms_ref.get_chemical_symbols()):
                fxr, fyr, fzr = forces_ref[atom_index]
                fxd, fyd, fzd = forces_dp[atom_index]
                fh_atom.write(
                    f"{frame_index:6d} {atom_index:6d} {symbol:2s} "
                    f"{fxr: .6f} {fyr: .6f} {fzr: .6f} "
                    f"{fxd: .6f} {fyd: .6f} {fzd: .6f} "
                    f"{energy_ref: .10f} {energy_dp: .10f}\n"
                )

    reports.append((label, frame_output, atom_output))

for label, frame_output, atom_output in reports:
    print(label)
    print(f"  {frame_output}")
    print(f"  {atom_output}")


To get the best performance, it is recommended to adjust the number of threads by setting the environment variables OMP_NUM_THREADS, DP_INTRA_OP_PARALLELISM_THREADS, and DP_INTER_OP_PARALLELISM_THREADS. See https://deepmd.rtfd.io/parallelism/ for more information.
I0000 00:00:1779103124.580042 4018143 mlir_graph_optimization_pass.cc:401] MLIR V1 optimization pass is not enabled
You can use the environment variable DP_INFER_BATCH_SIZE tocontrol the inference batch size (nframes * natoms). The default value is 1024.
You can use the environment variable DP_INFER_BATCH_SIZE tocontrol the inference batch size (nframes * natoms). The default value is 1024.
Evaluating graph.001.pb:   0%|          | 0/2000 [00:00<?, ?it/s]# warning: loc idx out of upper bound (ignored if warned for more than 10 times) 
# warning: loc idx out of upper bound (ignored if warned for more than 10 times) 
# warning: loc idx out of upper bound (ignored if warned for more than 10 times) 
# warning: loc idx out of uppe

graph.001.pb
  /Users/samuel/Desktop/postdoc_PhLAM/codes/deepmd_toolkit/model_analysis/outputs/inference/graph.001.pb/energies_forces.txt
  /Users/samuel/Desktop/postdoc_PhLAM/codes/deepmd_toolkit/model_analysis/outputs/inference/graph.001.pb/energies_forces_atoms.txt
graph.002.pb
  /Users/samuel/Desktop/postdoc_PhLAM/codes/deepmd_toolkit/model_analysis/outputs/inference/graph.002.pb/energies_forces.txt
  /Users/samuel/Desktop/postdoc_PhLAM/codes/deepmd_toolkit/model_analysis/outputs/inference/graph.002.pb/energies_forces_atoms.txt
graph.003.pb
  /Users/samuel/Desktop/postdoc_PhLAM/codes/deepmd_toolkit/model_analysis/outputs/inference/graph.003.pb/energies_forces.txt
  /Users/samuel/Desktop/postdoc_PhLAM/codes/deepmd_toolkit/model_analysis/outputs/inference/graph.003.pb/energies_forces_atoms.txt
graph.004.pb
  /Users/samuel/Desktop/postdoc_PhLAM/codes/deepmd_toolkit/model_analysis/outputs/inference/graph.004.pb/energies_forces.txt
  /Users/samuel/Desktop/postdoc_PhLAM/codes/deepmd_tool